# OSAI Project 1 — Colab v1 Baseline Reproduction

Pascal-VOC 21-class semantic segmentation, ResNet-50 + DeepLabV3+ (OS=16). main 브랜치 기준 v1 baseline 재현.

재현을 위해 다음 항목들을 수행해주셔야 합니다.

1. Drive에 폴더 생성: `MyDrive/osai/p1/submit/img/` (1000장 jpg)
2. WandB API key Colab Secret에 저장

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. 설정 변수

In [2]:
REPO_URL = "https://github.com/geniemo/osai.git"
BRANCH   = "main"
CONFIG   = "src/config/colab.yaml"
DRIVE    = "/content/drive/MyDrive/osai/p1"

import os
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True)
print(f"Repo: {REPO_URL} (branch={BRANCH})")
print(f"Config: {CONFIG}")
print(f"Drive: {DRIVE}")
assert os.path.exists(f"{DRIVE}/submit/img"), f"submit/img not found at {DRIVE}/submit/img"
n_imgs = len([f for f in os.listdir(f"{DRIVE}/submit/img") if f.endswith(".jpg")])
print(f"submit/img: {n_imgs} jpg files")

Repo: https://github.com/geniemo/osai.git (branch=main)
Config: src/config/colab.yaml
Drive: /content/drive/MyDrive/osai/p1
test_public: 1000 jpg files


## 2. 저장소 clone

In [3]:
%cd /content
!rm -rf osai
!git clone --branch {BRANCH} --depth 1 {REPO_URL} osai
%cd osai/p1

/content
Cloning into 'osai'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 147 (delta 14), reused 131 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 1.54 MiB | 4.97 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/osai/p1


## 3. uv 설치 + 의존성 sync

In [4]:
!pip install -q uv
!uv sync

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 65 packages in 0.98ms
Prepared 63 packages in 54.61s
Installed 63 packages in 350ms
 + annotated-types==0.7.0
 + certifi==2026.4.22
 + charset-normalizer==3.4.7
 + click==8.3.3
 + cuda-bindings==12.9.6
 + cuda-pathfinder==1.5.3
 + cuda-toolkit==12.8.1
 + filelock==3.29.0
 + fsspec==2026.3.0
 + gitdb==4.0.12
 + gitpython==3.1.47
 + idna==3.13
 + iniconfig==2.3.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + ml-dtypes==0.5.4
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==2.4.4
 + nvidia-cublas-cu12==12.8.4.1
 + nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cudnn-cu12==9.19.0.56
 + nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufile-cu12==1.13.1.3
 + nvidia-curand-cu12==10.3.9.90
 + nvidia-cusolver-cu12==11.7.3.90
 + nvidia-cusparse-cu12==12.5.8.93
 + nvidia-cusparselt-cu12==0.7.1
 + nvidia-nccl-cu12==2.28.9
 + nvidia-nvjitlink-cu12

## 4. GPU 확인

In [5]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
!uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

  File "<string>", line 1
    import torch;  print('Device:', torch.cuda.get_device_name(0));  print('Capability:', torch.cuda.get_device_capability(0));  print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB');  print('PyTorch:', torch.__version__)
IndentationError: unexpected indent


## 5. WandB 로그인

In [6]:
from google.colab import userdata
import os; os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

!uv run wandb login

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: g1nie (g1nie-sungkyunkwan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 6. Test images Colab 로컬로 복사

In [7]:
!mkdir -p submit/img
!cp -r {DRIVE}/submit/img/. submit/img/
# Drive submit/img/ → Colab submit/img/ 복사 (PDF 컨벤션)
!ls submit/img | wc -l
!ls submit/img | head -3

1000
000.jpg
001.jpg
002.jpg


## 7. 데이터 다운로드 (VOC + COCO)

In [8]:
!uv run python -m src.data.download --voc-root data/voc --coco-root data/coco

/usr/bin/python3: Error while finding module specification for 'src.data.download' (ModuleNotFoundError: No module named 'src')


## 8. COCO mask cache 사전 생성

95K instance mask를 PNG로 1회 생성

In [9]:
!uv run python -m src.build_coco_masks --coco-root data/coco --num-workers 4

/usr/bin/python3: Error while finding module specification for 'src.build_coco_masks' (ModuleNotFoundError: No module named 'src')


## 9. 학습 (Stage 1 + Stage 2 자동, ~10-12h)

ckpt가 Drive에 저장됨. 세션 끊기면 다시 시작 시 자동 resume.

In [10]:
!uv run python -m src.train --config {CONFIG}

/usr/bin/python3: Error while finding module specification for 'src.train' (ModuleNotFoundError: No module named 'src')


## 10. Validation mIoU 측정

최종 ckpt로 val mIoU + 21 class별 IoU 출력.

In [11]:
!uv run python -m src.eval --config {CONFIG} --ckpt {DRIVE}/checkpoints/model.pth

/usr/bin/python3: Error while finding module specification for 'src.eval' (ModuleNotFoundError: No module named 'src')


## 11. TTA Validation mIoU (선택, 실제 inference 성능)

Multi-scale + hflip TTA로 측정.

In [12]:
!uv run python -m src.eval_tta --config {CONFIG} --ckpt {DRIVE}/checkpoints/model.pth

/usr/bin/python3: Error while finding module specification for 'src.eval_tta' (ModuleNotFoundError: No module named 'src')


## 12. 추론 — submit/img → submit/pred (TTA)

PDF 재현 컨벤션과 동일. ~5–10분 (T4).

In [13]:
!mkdir -p submit/pred
!uv run python -m src.infer \
    --config {CONFIG} \
    --ckpt {DRIVE}/checkpoints/model.pth \
    --input submit/img \
    --output submit/pred

/usr/bin/python3: Error while finding module specification for 'src.infer' (ModuleNotFoundError: No module named 'src')


## 13. ONNX export (가중치 제거, 채점용 ONNX)

`model_structure.onnx` 생성 (~0.3MB, ≤10MB 채점 한도).

In [14]:
!uv run python -m src.export_onnx \
    --config {CONFIG} \
    --ckpt {DRIVE}/checkpoints/model.pth \
    --out {DRIVE}/model_structure.onnx

/usr/bin/python3: Error while finding module specification for 'src.export_onnx' (ModuleNotFoundError: No module named 'src')


## 14. FLOPs 측정

In [15]:
!uv run python -m src.measure_flops --onnx {DRIVE}/model_structure.onnx

/usr/bin/python3: Error while finding module specification for 'src.measure_flops' (ModuleNotFoundError: No module named 'src')


## 15. PNG zip 생성

In [16]:
!uv run python -m src.package_submission \
    --pred submit/pred \
    --out {DRIVE}/submission_pred.zip

/usr/bin/python3: Error while finding module specification for 'src.package_submission' (ModuleNotFoundError: No module named 'src')


## 16. 결과물 확인

In [17]:
!ls -la {DRIVE}/
!ls -la {DRIVE}/checkpoints/

total 8
drwx------ 2 root root 4096 Apr 25 05:17 checkpoints
drwx------ 3 root root 4096 Apr 25 05:21 input
total 0
